In [1]:
import os

import pandas as pd

import re

import itertools

import numpy as np

from scipy.stats import wilcoxon

from statistics import mean



# ==================== CONFIGURACIÓN DE RUTAS ====================

RESULTS_DIR = "../results"

PARSIMONY_SUMMARY_PATH = "../best_k_selection/parsimony_analysis/parsimony_summary.csv"

# Se cambia la ruta de salida a una carpeta específica para genes maestros

OUTPUT_DIR = "overlap_analysis"



K_VALUES = [25, 50, 75, 100, 500, 1000, 1500, 2500]

DATASETS_LIST = ["braaksc", "ceradsc", "cogdx"]

PREFIXES = ["Br", "Ce", "Co"] 



GREEN, RED, YELLOW, BLUE = '\033[92m', '\033[91m', '\033[93m', '\033[94m'

CYAN, MAGENTA, BOLD, RESET = '\033[96m', '\033[95m', '\033[1m', '\033[0m'



# ==================== FUNCIONES DE CARGA Y LÓGICA ====================



def setup_output_structure():

    # Se elimina la carpeta de plots ya que no se generarán diagramas de Venn

    folders = [OUTPUT_DIR, os.path.join(OUTPUT_DIR, "reports")]

    for folder in folders:

        if not os.path.exists(folder): os.makedirs(folder)



def get_winner_info(dataset, k):

    py_path = os.path.join(RESULTS_DIR, dataset, f"k_{k}", "best_analysis", f"{dataset}_best_dataset_k{k}.py")

    if not os.path.exists(py_path): return None

    with open(py_path, 'r', encoding='utf-8') as f: content = f.read()

    match = re.search(r"metodologia_ganador\s*=\s*['\"](.+?)['\"]", content)

    return match.group(1) if match else None



def get_dataset_path(dataset, k, winner_name):

    if not winner_name: return None

    base = f"genes-{dataset}"

    if "R2" in winner_name: fn = f"{base}-FR-{k}_train.csv"

    elif "R3" in winner_name: fn = f"{base}-resampling_FR-{k}_train.csv"

    else: fn = f"{base}_train.csv"

    return os.path.join(RESULTS_DIR, dataset, f"k_{k}", fn)



def load_metric_vector(dataset, k, method, metric="BA"):

    base = f"genes-{dataset}"

    suffix = f"_{base}-FR-{k}.csv" if "R2" in method else f"_{base}-resampling_FR-{k}.csv" if "R3" in method else f"_{base}_k{k}.csv"

    csv_path = os.path.join(RESULTS_DIR, dataset, f"k_{k}", f"test_{metric}{suffix}")

    if os.path.exists(csv_path):

        df = pd.read_csv(csv_path)

        col = method if method in df.columns else method.split("-")[0]

        if col in df.columns: return df[col].values[:10]

    return None



def calculate_win_loss_winner(dataset):

    scores_dict = {"BA": pd.DataFrame(), "F1": pd.DataFrame(), "PS": pd.DataFrame()}

    methods_found = {}

    for k in K_VALUES:

        meth = get_winner_info(dataset, k)

        if not meth: continue

        methods_found[f"K={k}"] = meth

        for m in ["BA", "F1", "PS"]:

            vec = load_metric_vector(dataset, k, meth, m)

            if vec is not None: scores_dict[m][f"K={k}"] = vec

    if scores_dict["BA"].empty: return None, None

    

    def get_net_wins(df):

        labels = df.columns

        wins = {l: 0 for l in labels}

        for i in range(len(labels)):

            for j in range(i + 1, len(labels)):

                col_i, col_j = df.iloc[:, i].values, df.iloc[:, j].values

                if not np.array_equal(col_i, col_j):

                    try:

                        _, p = wilcoxon(col_i, col_j)

                        if p < 0.05:

                            if mean(col_i) > mean(col_j): wins[labels[i]] += 1; wins[labels[j]] -= 1

                            else: wins[labels[j]] += 1; wins[labels[i]] -= 1

                    except: pass

        return wins



    w_ba, w_f1, w_ps = get_net_wins(scores_dict["BA"]), get_net_wins(scores_dict["F1"]), get_net_wins(scores_dict["PS"])

    total_wins = {k: w_ba[k] + w_f1[k] + w_ps[k] for k in w_ba.keys()}

    best_label = max(total_wins, key=total_wins.get)

    return int(best_label.replace("K=", "")), methods_found[best_label]



def load_genes(dataset, k, winner):

    path = get_dataset_path(dataset, k, winner)

    if not path or not os.path.exists(path): return set()

    return set(c for c in pd.read_csv(path, nrows=0).columns if c.lower() != 'target')



# ==================== EJECUCIÓN PRINCIPAL ====================



setup_output_structure()

if not os.path.exists(PARSIMONY_SUMMARY_PATH):

    print(f"{RED} Error: No existe {PARSIMONY_SUMMARY_PATH}{RESET}"); exit()



df_pars = pd.read_csv(PARSIMONY_SUMMARY_PATH)

dataset_options = {}

core_by_config = {}



print(f"\n{BOLD}{CYAN}  DEFINICIÓN DE OPCIONES POR DATASET{RESET}")

print("-" * 60)



for ds in DATASETS_LIST:

    row = df_pars[df_pars['Dataset'] == ds].iloc[0]

    kA, methA = int(row['K_elegido']), row['Metodologia']

    kB_raw, methB_raw = calculate_win_loss_winner(ds)

    

    if kB_raw == kA and methB_raw == methA:

        det_path = f"../best_k_selection/parsimony_analysis/metrics_per_k_{ds}.csv"

        df_det = pd.read_csv(det_path)

        pars_list = df_det[df_det['Elegido'] == 'SI'].sort_values(by='K')

        kB, methB = (int(pars_list.iloc[1]['K']), pars_list.iloc[1]['Metodologia']) if len(pars_list) > 1 else (25, get_winner_info(ds, 25))

    else:

        kB, methB = kB_raw, methB_raw

        

    dataset_options[ds] = {'A': {'k': kA, 'meth': methA}, 'B': {'k': kB, 'meth': methB}}

    

    print(f"{BOLD}{ds.upper()}:{RESET}")

    print(f"  {GREEN}Opción A (Parsimonia):{RESET} K={kA} | {methA}")

    print(f"  {YELLOW}Opción B (Win-Loss):  {RESET} K={kB} | {methB}")



combos = list(itertools.product(['A', 'B'], repeat=3))



for combo in combos:

    config_id = "-".join([f"{p}{opt}" for p, opt in zip(PREFIXES, combo)])

    print(f"\n{BOLD} PROCESANDO: {config_id}{RESET}")

    

    gene_sets = {ds: load_genes(ds, dataset_options[ds][combo[i]]['k'], dataset_options[ds][combo[i]]['meth']) for i, ds in enumerate(DATASETS_LIST)}

    core = set.intersection(*gene_sets.values())

    core_by_config[config_id] = core



    rep_lines = [f"CONFIG: {config_id}\n", "-"*30 + "\n"]

    for i, ds in enumerate(DATASETS_LIST):

        p = dataset_options[ds][combo[i]]

        rep_lines.append(f"{ds.upper()} ({PREFIXES[i]}{combo[i]}): K={p['k']} ({p['meth']})\n")

    rep_lines.append(f"\nMAESTROS (TRIPLE OVERLAP): {len(core)}\n{', '.join(sorted(list(core)))}\n")

    

    report_path = os.path.join(OUTPUT_DIR, "reports", f"report_{config_id}.txt")

    with open(report_path, "w", encoding="utf-8") as f: 

        f.writelines(rep_lines)

    print(f"   {MAGENTA} Reporte detallado guardado en: {report_path}{RESET}")



# ==================== ANÁLISIS DE EXCLUSIVIDAD  ====================



print("\n" + "="*80)

print(f"{BOLD}{CYAN} COMPARATIVA FINAL: EXCLUSIVIDAD DE GENES MAESTROS{RESET}")

print("="*80)



excl_report = ["INFORME DE EXCLUSIVIDAD GENÓMICA\n", "="*40 + "\n"]



for cid, core_genes in core_by_config.items():

    parts = cid.split("-")

    desc_list = []

    for i, p_code in enumerate(parts):

        ds_name = DATASETS_LIST[i]

        opt_letter = p_code[-1]

        k_val = dataset_options[ds_name][opt_letter]['k']

        desc_list.append(f"{p_code} (K={k_val})")

    

    desc_str = " | ".join(desc_list)

    

    others_union = set().union(*[v for k, v in core_by_config.items() if k != cid])

    exclusivos = core_genes - others_union

    

    print(f"\n{BOLD} CONFIGURACIÓN: {cid}{RESET}")

    print(f"    {BLUE}Definición:{RESET} {desc_str}")

    print(f"    {BOLD}Resultados:{RESET} {len(core_genes)} maestros | {GREEN}{len(exclusivos)} exclusivos{RESET}")

    

    excl_report.append(f"CONFIG {cid} [{desc_str}]:\n")

    excl_report.append(f"Total maestros: {len(core_genes)} | Exclusivos: {len(exclusivos)}\n")

    

    if exclusivos:

        excl_list = sorted(list(exclusivos))

        print(f"    {GREEN}Genes exclusivos:{RESET}")

        print(f"    {', '.join(excl_list)}")

        excl_report.append(f"    LISTA COMPLETA: {', '.join(excl_list)}\n\n")

    else:

        print(f"    {RED}Sin genes maestros exclusivos.{RESET}")

        excl_report.append("    Sin genes exclusivos.\n\n")



final_summary_path = os.path.join(OUTPUT_DIR, "reports", "master_exclusivity_summary.txt")

with open(final_summary_path, "w", encoding="utf-8") as f: 

    f.writelines(excl_report)



print(f"\n{BOLD}{MAGENTA} Resumen de exclusividad final guardado en: {final_summary_path}{RESET}")

print(f"\n{BOLD}{GREEN} PROCESO COMPLETADO DINÁMICAMENTE.{RESET}")


  DEFINICIÓN DE OPCIONES POR DATASET
------------------------------------------------------------
BRAAKSC:
  Opción A (Parsimonia): K=50 | BorderlineSMOTE_RF-R2_FR
  Opción B (Win-Loss):   K=75 | ROS_RF-R2_FR
CERADSC:
  Opción A (Parsimonia): K=1500 | NCR_RF-R3_Resampl
  Opción B (Win-Loss):   K=25 | AllKNN_RF-R3_Resampl
COGDX:
  Opción A (Parsimonia): K=1000 | ROS_RF-R2_FR
  Opción B (Win-Loss):   K=75 | Tomek_RF-R2_FR

 PROCESANDO: BrA-CeA-CoA
    Reporte detallado guardado en: overlap_analysis\reports\report_BrA-CeA-CoA.txt

 PROCESANDO: BrA-CeA-CoB
    Reporte detallado guardado en: overlap_analysis\reports\report_BrA-CeA-CoB.txt

 PROCESANDO: BrA-CeB-CoA
    Reporte detallado guardado en: overlap_analysis\reports\report_BrA-CeB-CoA.txt

 PROCESANDO: BrA-CeB-CoB
    Reporte detallado guardado en: overlap_analysis\reports\report_BrA-CeB-CoB.txt

 PROCESANDO: BrB-CeA-CoA
    Reporte detallado guardado en: overlap_analysis\reports\report_BrB-CeA-CoA.txt

 PROCESANDO: BrB-CeA-CoB
    

In [2]:
import os
import pandas as pd
import re
import itertools
import numpy as np
from scipy.stats import wilcoxon
from statistics import mean
from collections import defaultdict

# ==================== CONFIGURACIÓN DE RUTAS ====================
# Rutas para RESCATAR datos
RESULTS_DIR = "../results"
PARSIMONY_SUMMARY_PATH = "../best_k_selection/parsimony_analysis/parsimony_summary.csv"

# Rutas para CREAR/GUARDAR datos (Ruta actualizada)
OUTPUT_DIR = "overlap_analysis"

K_VALUES = [25, 50, 75, 100, 500, 1000, 1500, 2500]
DATASETS_LIST = ["braaksc", "ceradsc", "cogdx"]
PREFIXES = ["Br", "Ce", "Co"] 

# Formatos de consola
GREEN, RED, YELLOW, BLUE = '\033[92m', '\033[91m', '\033[93m', '\033[94m'
CYAN, MAGENTA, BOLD, RESET = '\033[96m', '\033[95m', '\033[1m', '\033[0m'
PRINT_LIMIT = 80

# ==================== GESTIÓN DE DIRECTORIOS ====================

def setup_output_structure():
    """Crea la estructura de carpetas localmente sin incluir plots."""
    folders = [
        OUTPUT_DIR, 
        os.path.join(OUTPUT_DIR, "reports")
    ]
    for folder in folders:
        if not os.path.exists(folder):
            os.makedirs(folder, exist_ok=True)
    print(f"{GREEN} Estructura de carpetas local lista en: {OUTPUT_DIR}{RESET}")

# ==================== FUNCIONES DE CARGA Y LÓGICA ====================

def get_winner_info(dataset, k):
    py_path = os.path.join(RESULTS_DIR, dataset, f"k_{k}", "best_analysis", f"{dataset}_best_dataset_k{k}.py")
    if not os.path.exists(py_path): return None
    with open(py_path, 'r', encoding='utf-8') as f: content = f.read()
    match = re.search(r"metodologia_ganador\s*=\s*['\"](.+?)['\"]", content)
    return match.group(1) if match else None

def get_dataset_path(dataset, k, winner_name):
    if not winner_name: return None
    base = f"genes-{dataset}"
    if "R2" in winner_name: fn = f"{base}-FR-{k}_train.csv"
    elif "R3" in winner_name: fn = f"{base}-resampling_FR-{k}_train.csv"
    else: fn = f"{base}_train.csv"
    return os.path.join(RESULTS_DIR, dataset, f"k_{k}", fn)

def load_metric_vector(dataset, k, method, metric="BA"):
    base = f"genes-{dataset}"
    suffix = f"_{base}-FR-{k}.csv" if "R2" in method else f"_{base}-resampling_FR-{k}.csv" if "R3" in method else f"_{base}_k{k}.csv"
    csv_path = os.path.join(RESULTS_DIR, dataset, f"k_{k}", f"test_{metric}{suffix}")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        col = method if method in df.columns else method.split("-")[0]
        if col in df.columns: return df[col].values[:10]
    return None

def calculate_win_loss_winner(dataset):
    scores_dict = {"BA": pd.DataFrame(), "F1": pd.DataFrame(), "PS": pd.DataFrame()}
    methods_found = {}
    for k in K_VALUES:
        meth = get_winner_info(dataset, k)
        if not meth: continue
        methods_found[f"K={k}"] = meth
        for m in ["BA", "F1", "PS"]:
            vec = load_metric_vector(dataset, k, meth, m)
            if vec is not None: scores_dict[m][f"K={k}"] = vec
    if scores_dict["BA"].empty: return None, None
    
    def get_net_wins(df):
        labels = df.columns
        wins = {l: 0 for l in labels}
        for i in range(len(labels)):
            for j in range(i + 1, len(labels)):
                col_i, col_j = df.iloc[:, i].values, df.iloc[:, j].values
                if not np.array_equal(col_i, col_j):
                    try:
                        _, p = wilcoxon(col_i, col_j)
                        if p < 0.05:
                            if mean(col_i) > mean(col_j): wins[labels[i]] += 1; wins[labels[j]] -= 1
                            else: wins[labels[j]] += 1; wins[labels[i]] -= 1
                    except: pass
        return wins

    w_ba, w_f1, w_ps = get_net_wins(scores_dict["BA"]), get_net_wins(scores_dict["F1"]), get_net_wins(scores_dict["PS"])
    total_wins = {k: w_ba[k] + w_f1[k] + w_ps[k] for k in w_ba.keys()}
    best_label = max(total_wins, key=total_wins.get)
    return int(best_label.replace("K=", "")), methods_found[best_label]

def load_genes(dataset, k, winner):
    path = get_dataset_path(dataset, k, winner)
    if not path or not os.path.exists(path): return set()
    return set(c for c in pd.read_csv(path, nrows=0).columns if c.lower() != 'target')

# ==================== EJECUCIÓN PRINCIPAL ====================

setup_output_structure()

if not os.path.exists(PARSIMONY_SUMMARY_PATH):
    print(f"{RED} Error: No existe el resumen de parsimonia en: {PARSIMONY_SUMMARY_PATH}{RESET}"); exit()

df_pars = pd.read_csv(PARSIMONY_SUMMARY_PATH)
dataset_options = {}
core_by_config = {}

print(f"\n{BOLD}{CYAN}  CONFIGURANDO OPCIONES (A: PARSIMONIA | B: WIN-LOSS){RESET}")

for ds in DATASETS_LIST:
    row = df_pars[df_pars['Dataset'] == ds].iloc[0]
    kA, methA = int(row['K_elegido']), row['Metodologia']
    kB_raw, methB_raw = calculate_win_loss_winner(ds)
    
    if kB_raw == kA and methB_raw == methA:
        det_path = f"../best_k_selection/parsimony_analysis/metrics_per_k_{ds}.csv"
        if os.path.exists(det_path):
            df_det = pd.read_csv(det_path)
            pars_list = df_det[df_det['Elegido'] == 'SI'].sort_values(by='K')
            kB, methB = (int(pars_list.iloc[1]['K']), pars_list.iloc[1]['Metodologia']) if len(pars_list) > 1 else (25, get_winner_info(ds, 25))
        else:
            kB, methB = (25, get_winner_info(ds, 25)) if kA != 25 else (50, get_winner_info(ds, 50))
    else:
        kB, methB = kB_raw, methB_raw
        
    dataset_options[ds] = {'A': {'k': kA, 'meth': methA}, 'B': {'k': kB, 'meth': methB}}
    print(f"   {ds.upper()}: A(K={kA}) | B(K={kB})")

combos = list(itertools.product(['A', 'B'], repeat=3))

for combo in combos:
    config_id = "-".join([f"{p}{opt}" for p, opt in zip(PREFIXES, combo)])
    print(f"\n{BOLD} PROCESANDO: {config_id}{RESET}")
    
    gene_sets = {ds: load_genes(ds, dataset_options[ds][combo[i]]['k'], dataset_options[ds][combo[i]]['meth']) for i, ds in enumerate(DATASETS_LIST)}
    core = set.intersection(*gene_sets.values())
    core_by_config[config_id] = core

    rep_lines = [f"CONFIG: {config_id}\n", "-"*40 + "\n"]
    for i, ds in enumerate(DATASETS_LIST):
        p = dataset_options[ds][combo[i]]
        rep_lines.append(f"{ds.upper()} ({PREFIXES[i]}{combo[i]}): K={p['k']} ({p['meth']})\n")
    rep_lines.append(f"\nMAESTROS (TRIPLE OVERLAP): {len(core)}\n{', '.join(sorted(list(core)))}\n")
    
    report_path = os.path.join(OUTPUT_DIR, "reports", f"report_{config_id}.txt")
    with open(report_path, "w", encoding="utf-8") as f: 
        f.writelines(rep_lines)
    print(f"   {MAGENTA} Reporte detallado guardado en: {report_path}{RESET}")

# ==============================================================================
# ANÁLISIS DE PATRONES DE PRESENCIA Y EXPORTACIÓN CSV
# ==============================================================================

print("\n" + "="*80)
print(f"{BOLD}{CYAN} ANÁLISIS DE PATRONES Y RANKING DE ESTABILIDAD{RESET}")
print("="*80)

all_core_genes = set().union(*core_by_config.values())

gene_presence = defaultdict(list)
for cid, genes in core_by_config.items():
    for g in genes:
        gene_presence[g].append(cid)

total_configs = len(core_by_config)
not_in_all = {g: cfgs for g, cfgs in gene_presence.items() if len(cfgs) < total_configs}
estables_count = len(all_core_genes) - len(not_in_all)

# --- CREACIÓN DEL CSV DE RANKING DETALLADO ---
csv_data = []
ds_map = {"Br": "braaksc", "Ce": "ceradsc", "Co": "cogdx"}

for gene, configs in gene_presence.items():
    unique_components = set()
    for cid in configs:
        for component in cid.split("-"):
            unique_components.add(component)
    
    formatted_details = []
    for comp in sorted(list(unique_components)):
        prefix = comp[:2]
        opt = comp[-1]
        ds_name = ds_map[prefix]
        k_val = dataset_options[ds_name][opt]['k']
        formatted_details.append(f"{comp} [K={k_val}]")
    
    csv_data.append({
        "Gene": gene,
        "Appearances": len(configs),
        "Stability_%": (len(configs)/total_configs)*100,
        "Dataset_Details": ", ".join(formatted_details)
    })

df_ranking = pd.DataFrame(csv_data).sort_values(by="Appearances", ascending=False)
csv_output_path = os.path.join(OUTPUT_DIR, "master_genes_stability_ranking.csv")
df_ranking.to_csv(csv_output_path, index=False)
print(f"\n   {MAGENTA} Ranking de estabilidad guardado en: {csv_output_path}{RESET}")

# --- REPORTE TXT Y CONSOLA ---
print(f"\n Total genes maestros volátiles: {YELLOW}{len(not_in_all)}{RESET}")
print(f" Total genes maestros estables: {GREEN}{estables_count}{RESET}\n")

pattern_report = [
    "INFORME DE ESTABILIDAD Y PATRONES DE GENES MAESTROS\n",
    f"Genes Estables (en las {total_configs} configs): {estables_count}\n",
    f"Genes Volátiles: {len(not_in_all)}\n",
    "="*80 + "\n"
]

pattern_groups = defaultdict(list)
for gene, configs in not_in_all.items():
    pattern_groups[tuple(sorted(configs))].append(gene)

sorted_patterns = sorted(pattern_groups.items(), key=lambda x: len(x[0]), reverse=True)

for configs, genes in sorted_patterns:
    header = f" Aparecen en {len(configs)} configuraciones: {', '.join(configs)}"
    print("-" * 70)
    print(f"{BOLD}{header}{RESET}")
    print(f" Genes ({len(genes)}):")
    
    genes_sorted = sorted(genes)
    print("   " + ", ".join(genes_sorted[:PRINT_LIMIT]))
    if len(genes_sorted) > PRINT_LIMIT:
        print(f"   ... y {len(genes_sorted) - PRINT_LIMIT} más")
    
    pattern_report.append(f"\n{header}\n")
    pattern_report.append(f"Genes ({len(genes)}): {', '.join(genes_sorted)}\n")

pattern_output_path = os.path.join(OUTPUT_DIR, "reports", "pattern_stability_analysis.txt")
with open(pattern_output_path, "w", encoding="utf-8") as f:
    f.writelines(pattern_report)
print(f"\n   {MAGENTA} Informe de patrones guardado en: {pattern_output_path}{RESET}")

print(f"\n{BOLD}{GREEN} PROCESO COMPLETADO DINÁMICAMENTE.{RESET}")

 Estructura de carpetas local lista en: overlap_analysis

  CONFIGURANDO OPCIONES (A: PARSIMONIA | B: WIN-LOSS)
   BRAAKSC: A(K=50) | B(K=75)
   CERADSC: A(K=1500) | B(K=25)
   COGDX: A(K=1000) | B(K=75)

 PROCESANDO: BrA-CeA-CoA
    Reporte detallado guardado en: overlap_analysis\reports\report_BrA-CeA-CoA.txt

 PROCESANDO: BrA-CeA-CoB
    Reporte detallado guardado en: overlap_analysis\reports\report_BrA-CeA-CoB.txt

 PROCESANDO: BrA-CeB-CoA
    Reporte detallado guardado en: overlap_analysis\reports\report_BrA-CeB-CoA.txt

 PROCESANDO: BrA-CeB-CoB
    Reporte detallado guardado en: overlap_analysis\reports\report_BrA-CeB-CoB.txt

 PROCESANDO: BrB-CeA-CoA
    Reporte detallado guardado en: overlap_analysis\reports\report_BrB-CeA-CoA.txt

 PROCESANDO: BrB-CeA-CoB
    Reporte detallado guardado en: overlap_analysis\reports\report_BrB-CeA-CoB.txt

 PROCESANDO: BrB-CeB-CoA
    Reporte detallado guardado en: overlap_analysis\reports\report_BrB-CeB-CoA.txt

 PROCESANDO: BrB-CeB-CoB
    Rep

In [3]:
import os
import pandas as pd
import re
import itertools
import numpy as np
from scipy.stats import wilcoxon
from statistics import mean
from collections import defaultdict

# ==================== CONFIG ====================

RESULTS_DIR = "../results"
PARSIMONY_SUMMARY_PATH = "../best_k_selection/parsimony_analysis/parsimony_summary.csv"
OUTPUT_DIR = "overlap_analysis"

K_VALUES = [25, 50, 75, 100, 500, 1000, 1500, 2500]
DATASETS = ["braaksc", "ceradsc", "cogdx"]
PREFIXES = ["Br", "Ce", "Co"]

MIN_STABILITY = 2  #  CLAVE: elimina genes volátiles (1 sola config)

# ==================== UTILS ====================

def setup_dirs():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "reports"), exist_ok=True)

def get_winner_info(dataset, k):
    path = os.path.join(RESULTS_DIR, dataset, f"k_{k}", "best_analysis", f"{dataset}_best_dataset_k{k}.py")
    if not os.path.exists(path): return None
    txt = open(path, encoding="utf-8").read()
    m = re.search(r"metodologia_ganador\s*=\s*['\"](.+?)['\"]", txt)
    return m.group(1) if m else None

def get_dataset_path(dataset, k, method):
    base = f"genes-{dataset}"
    if "R2" in method:
        fn = f"{base}-FR-{k}_train.csv"
    elif "R3" in method:
        fn = f"{base}-resampling_FR-{k}_train.csv"
    else:
        fn = f"{base}_train.csv"
    return os.path.join(RESULTS_DIR, dataset, f"k_{k}", fn)

def load_genes(dataset, k, method):
    path = get_dataset_path(dataset, k, method)
    if not os.path.exists(path): return set()
    cols = pd.read_csv(path, nrows=0).columns
    return set(c for c in cols if c.lower() != "target")

# ==================== WIN-LOSS ====================

def load_metric(dataset, k, method, metric):
    base = f"genes-{dataset}"
    if "R2" in method:
        suffix = f"_{base}-FR-{k}.csv"
    elif "R3" in method:
        suffix = f"_{base}-resampling_FR-{k}.csv"
    else:
        suffix = f"_{base}_k{k}.csv"

    path = os.path.join(RESULTS_DIR, dataset, f"k_{k}", f"test_{metric}{suffix}")
    if not os.path.exists(path): return None

    df = pd.read_csv(path)
    col = method if method in df.columns else method.split("-")[0]
    return df[col].values[:10] if col in df.columns else None

def win_loss(dataset):
    scores = {"BA": pd.DataFrame(), "F1": pd.DataFrame(), "PS": pd.DataFrame()}
    methods = {}

    for k in K_VALUES:
        m = get_winner_info(dataset, k)
        if not m: continue
        methods[f"K={k}"] = m
        for metric in scores:
            v = load_metric(dataset, k, m, metric)
            if v is not None:
                scores[metric][f"K={k}"] = v

    def net(df):
        labs = df.columns
        wins = {l: 0 for l in labs}
        for i in range(len(labs)):
            for j in range(i+1, len(labs)):
                a, b = df.iloc[:, i], df.iloc[:, j]
                if not np.array_equal(a, b):
                    try:
                        _, p = wilcoxon(a, b)
                        if p < 0.05:
                            if mean(a) > mean(b):
                                wins[labs[i]] += 1; wins[labs[j]] -= 1
                            else:
                                wins[labs[j]] += 1; wins[labs[i]] -= 1
                    except:
                        pass
        return wins

    w = {k: net(scores[k]) for k in scores}
    total = {k: w["BA"][k] + w["F1"][k] + w["PS"][k] for k in w["BA"]}
    best = max(total, key=total.get)
    return int(best.replace("K=", "")), methods[best]

# ==================== MAIN ====================

setup_dirs()
df_pars = pd.read_csv(PARSIMONY_SUMMARY_PATH)

# Selección A/B
options = {}
for ds in DATASETS:
    row = df_pars[df_pars["Dataset"] == ds].iloc[0]
    kA, mA = int(row["K_elegido"]), row["Metodologia"]
    kB, mB = win_loss(ds)
    options[ds] = {"A": (kA, mA), "B": (kB, mB)}

# combinaciones
combos = list(itertools.product(["A", "B"], repeat=3))

# ==================== RECOLECTAR ====================

config_results = {}
core_by_config = defaultdict(set)

for combo in combos:
    cid = "-".join([f"{p}{o}" for p, o in zip(PREFIXES, combo)])

    genes = {}
    for i, ds in enumerate(DATASETS):
        k, m = options[ds][combo[i]]
        genes[ds] = load_genes(ds, k, m)

    config_results[cid] = genes
    core_by_config[cid] = set.intersection(*genes.values())

# ==================== ESTABILIDAD GLOBAL ====================

gene_count = defaultdict(int)
for genes in core_by_config.values():
    for g in genes:
        gene_count[g] += 1

stable_genes = {g for g, c in gene_count.items() if c >= MIN_STABILITY}

# ==================== ANÁLISIS CLÍNICO LIMPIO ====================

def clinical_sets(genes):
    return {
        "Resiliencia Cognitiva (Braak+Cerad, NO Cogdx)": (genes["braaksc"] & genes["ceradsc"]) - genes["cogdx"],
        "Deterioro sin Amiloide (Braak+Cogdx, NO Cerad)": (genes["braaksc"] & genes["cogdx"]) - genes["ceradsc"],
        "Deterioro sin Tau (Cerad+Cogdx, NO Braak)": (genes["ceradsc"] & genes["cogdx"]) - genes["braaksc"],
    }

# aplicar filtro estabilidad
clinical_clean = {}
for cid, genes in config_results.items():
    raw = clinical_sets(genes)
    clinical_clean[cid] = {k: v & stable_genes for k, v in raw.items()}

# ==================== PRINT ====================

print("="*80)
print(" GENES DE INTERSECCIONES CLÍNICAS POR CONFIGURACIÓN")
print("="*80)

for cid in clinical_clean:
    print("\n" + "-"*80)
    print(f" CONFIGURACIÓN: {cid}")
    print("-"*80)

    for name, genes in clinical_clean[cid].items():
        print(f"\n    {name} → {len(genes)} genes")
        if genes:
            print("      " + ", ".join(sorted(genes)))
        else:
            print("      Ninguno")

# ==================== EXCLUSIVIDAD REAL ====================

print("\n" + "="*80)
print(" GENES EXCLUSIVOS DE CADA CONFIGURACIÓN (FILTRADO)")
print("="*80)

for cat in list(clinical_clean.values())[0].keys():
    print("\n" + "-"*70)
    print(f" {cat}")
    print("-"*70)

    genes_by_cfg = {c: clinical_clean[c][cat] for c in clinical_clean}

    for cfg, genes in genes_by_cfg.items():
        other = set().union(*[g for k, g in genes_by_cfg.items() if k != cfg])
        excl = genes - other

        print(f"\n Exclusivos de {cfg} → {len(excl)} genes")
        if excl:
            print("   " + ", ".join(sorted(excl)))
        else:
            print("   Ninguno")

print("\n ANÁLISIS LIMPIO COMPLETADO")

 GENES DE INTERSECCIONES CLÍNICAS POR CONFIGURACIÓN

--------------------------------------------------------------------------------
 CONFIGURACIÓN: BrA-CeA-CoA
--------------------------------------------------------------------------------

    Resiliencia Cognitiva (Braak+Cerad, NO Cogdx) → 0 genes
      Ninguno

    Deterioro sin Amiloide (Braak+Cogdx, NO Cerad) → 0 genes
      Ninguno

    Deterioro sin Tau (Cerad+Cogdx, NO Braak) → 22 genes
      AIG1, AMPD2, CAMK2G, CMSS1, CREBL2, CRELD2, CXXC1, EXOSC1, HMBOX1, LAPTM4B, LRRC34, PDHA1, PDZD11, PRPF4, PYCR3, RABGGTA, SLC6A12, SLC6A9, ST3GAL3, STAU1, STX10, TAF1D

--------------------------------------------------------------------------------
 CONFIGURACIÓN: BrA-CeA-CoB
--------------------------------------------------------------------------------

    Resiliencia Cognitiva (Braak+Cerad, NO Cogdx) → 21 genes
      AGPAT1, ANKRD30B, BBOF1, CAB39L, CAPZA1, CASP9, CLASRP, DTX1, ERP44, IP6K2, KMT2D, LRCH4, ME2, MPV17L2, MXI1, OBSCN